## Setup and Data Generation

In [1]:
# Import os library to access environment variables and operating system functionalities
import os

# Set environment variable to avoid duplicate library errors, specifically for OpenMP
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
# Set TensorFlow log level to '2' to suppress most messages except warnings and errors
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import sys
sys.path.append("../utilities")

# Import necessary libraries for deep learning and numerical computation
import tensorflow as tf
import numpy as np
import DataGenerator as DG
import pysindy as ps
import time
import pandas as pd
import matplotlib.pyplot as plt

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# Specify the model type
MY_MODEL = 'lorenz_96'  # discrete sine Gordon model

LORENZ96_DIM = 100  # You can change this value to your desired number of elements
# Generate N random numbers from a normal distribution (mean=0, std=1)

num_traj = 1
data_initial_conditions = DG.DataGenerator.generate_initial_conditions(LORENZ96_DIM, num_traj)

# Data generation parameters
data_T = 60.0       # Length of the data (T)
data_dt = 0.001     # Resolution of the data (dt)
noise_level = 0.01  # 1% noise level

# Create DataGenerator with noise_level parameter
myDG = DG.DataGenerator(data_initial_conditions, T=data_T, dt=data_dt, 
                        noise_level=noise_level, derivative_mode="exact")

# Generate dataset (noise will be added automatically by DataGenerator)
t_arr, x_train, dx_train, guess_highest_order_polynomial = myDG.generate_dataset_by_model_name(MY_MODEL, method='DOP853')

print(f"Noise level: {noise_level*100}%")
print("Guess highest order polynomial:", guess_highest_order_polynomial)
print("Training input shape:", x_train.shape)
print("Training output shape:", dx_train.shape)

Noise level: 1.0%
Guess highest order polynomial: 2
Training input shape: (60000, 100)
Training output shape: (60000, 100)


In [3]:
# Prepare training data
training_input, training_output = x_train, dx_train
training_input_for_sim = training_input.copy()  # Saving a copy of training input for simulation

print("Training input shape:", training_input.shape)
print("Training output shape:", training_output.shape)

Training input shape: (60000, 100)
Training output shape: (60000, 100)


## Define Optimizers and Hyperparameters

In [4]:
# Define optimizers and their hyperparameter ranges
optimizer_configs = [
    {
        'name': 'STLSQ',
        'thresholds': [0.001, 0.005, 0.01],
        'create_optimizer': lambda threshold: ps.STLSQ(threshold=threshold)
    },
    {
        'name': 'LASSO',
        'alphas': [1e-5, 1e-4, 1e-3, 1e-2],
        'create_optimizer': lambda alpha: ps.STLSQ(alpha=alpha, threshold=0.0)
    },
    {
        'name': 'Ridge',
        'alphas': [1e-5, 1e-4, 1e-3, 1e-2],
        'create_optimizer': lambda alpha: ps.STLSQ(alpha=alpha, ridge_kw={'alpha': alpha}, threshold=0.0)
    },
    {
        'name': 'SR3',
        'thresholds': [0.001, 0.005, 0.01],
        'create_optimizer': lambda threshold: ps.SR3(threshold=threshold, thresholder='l0')
    }
]

print("Optimizer configurations defined")

Optimizer configurations defined


## Ablation Study Function

In [5]:
def evaluate_model(model, training_input, training_output, t_arr, optimizer_name, hyperparameter_value):
    """
    Evaluate a SINDy model and return comprehensive metrics.
    """
    results = {
        'optimizer': optimizer_name,
        'hyperparameter': hyperparameter_value
    }
    
    # 1. Coefficient Analysis and Sparsity
    coefficients = model.coefficients()
    total_coefficients = coefficients.size
    nonzero_coefficients = np.count_nonzero(coefficients)
    sparsity = 1 - (nonzero_coefficients / total_coefficients)
    
    results['total_coefficients'] = total_coefficients
    results['nonzero_coefficients'] = nonzero_coefficients
    results['sparsity'] = sparsity
    
    # 2. Training Data Prediction Error
    dx_pred = model.predict(training_input)
    dx_true = training_output
    
    mse = np.mean((dx_pred - dx_true)**2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(dx_pred - dx_true))
    max_error = np.max(np.abs(dx_pred - dx_true))
    
    results['derivative_mse'] = mse
    results['derivative_rmse'] = rmse
    results['derivative_mae'] = mae
    results['derivative_max_error'] = max_error
    
    # 3. Trajectory Prediction Error
    try:
        x_sim = model.simulate(training_input[0], t_arr)
        
        traj_mse = np.mean((x_sim - training_input)**2)
        traj_rmse = np.sqrt(traj_mse)
        traj_mae = np.mean(np.abs(x_sim - training_input))
        
        results['trajectory_mse'] = traj_mse
        results['trajectory_rmse'] = traj_rmse
        results['trajectory_mae'] = traj_mae
    except Exception as e:
        print(f"Simulation failed for {optimizer_name} with hyperparameter {hyperparameter_value}: {e}")
        results['trajectory_mse'] = np.nan
        results['trajectory_rmse'] = np.nan
        results['trajectory_mae'] = np.nan
    
    # 4. Model Complexity (using PySINDy's built-in method)
    try:
        # PySINDy provides complexity as the number of terms in the model
        complexity = model.complexity
        results['model_complexity'] = complexity
    except:
        # If complexity attribute is not available, use nonzero coefficients
        results['model_complexity'] = nonzero_coefficients
    
    # Active terms per equation
    active_terms_per_eq = np.count_nonzero(coefficients, axis=1)
    avg_active_terms = np.mean(active_terms_per_eq)
    results['avg_active_terms_per_equation'] = avg_active_terms
    
    return results

## Run Ablation Study

In [ ]:
# Store all results
all_results = []

# Feature library (same for all optimizers)
feature_library = ps.PolynomialLibrary(degree=2)

print("Starting ablation study...\n")
print("="*80)

for config in optimizer_configs:
    optimizer_name = config['name']
    print(f"\nTesting {optimizer_name}...")
    print("-"*80)
    
    # Get hyperparameter list
    if 'thresholds' in config:
        hyperparameters = config['thresholds']
        param_name = 'threshold'
    else:
        hyperparameters = config['alphas']
        param_name = 'alpha'
    
    for hyperparam in hyperparameters:
        print(f"  Testing {param_name}={hyperparam}...")
        
        try:
            # Create optimizer
            optimizer = config['create_optimizer'](hyperparam)
            
            # Create and fit model
            model = ps.SINDy(feature_library=feature_library, optimizer=optimizer)
            
            start_time = time.time()
            model.fit(training_input, t=t_arr, x_dot=training_output)
            training_time = time.time() - start_time
            
            # Evaluate model
            results = evaluate_model(model, training_input, training_output, t_arr, 
                                    optimizer_name, hyperparam)
            results['training_time'] = training_time
            
            all_results.append(results)
            
            print(f"    Sparsity: {results['sparsity']:.4f}, "
                  f"RMSE: {results['derivative_rmse']:.6f}, "
                  f"Training time: {training_time:.4f}s")
            
        except Exception as e:
            print(f"    ERROR: {e}")
            continue

print("\n" + "="*80)
print("Ablation study complete!")
print(f"Total experiments run: {len(all_results)}")

Starting ablation study...


Testing STLSQ...
--------------------------------------------------------------------------------
  Testing threshold=0.001...
Simulation failed for STLSQ with hyperparameter 0.001: `t_eval` must be 1-dimensional.
    Sparsity: 0.0704, RMSE: 0.216632, Training time: 36372.6411s
  Testing threshold=0.005...
Simulation failed for STLSQ with hyperparameter 0.005: `t_eval` must be 1-dimensional.
    Sparsity: 0.3831, RMSE: 0.217975, Training time: 13051.4780s
  Testing threshold=0.01...


/opt/anaconda3/envs/sreinet/lib/python3.10/site-packages/pysindy/optimizers/stlsq.py:269: ConvergenceWarning: STLSQ did not converge after {self.max_iter} iterations.
  warnings.warn(


Simulation failed for STLSQ with hyperparameter 0.01: `t_eval` must be 1-dimensional.
    Sparsity: 0.7207, RMSE: 0.230071, Training time: 4554.7107s

Testing LASSO...
--------------------------------------------------------------------------------
  Testing alpha=1e-05...
Simulation failed for LASSO with hyperparameter 1e-05: `t_eval` must be 1-dimensional.
    Sparsity: 0.0000, RMSE: 0.216624, Training time: 7444.7943s
  Testing alpha=0.0001...
Simulation failed for LASSO with hyperparameter 0.0001: `t_eval` must be 1-dimensional.
    Sparsity: 0.0000, RMSE: 0.216624, Training time: 8091.6483s
  Testing alpha=0.001...


## Results Analysis and Visualization

In [ ]:
# Convert results to DataFrame
df_results = pd.DataFrame(all_results)

# Display summary
print("\nResults Summary:")
print("="*80)
display(df_results)

In [ ]:
# Statistical summary by optimizer
print("\nStatistical Summary by Optimizer:")
print("="*80)
summary_stats = df_results.groupby('optimizer').agg({
    'sparsity': ['mean', 'std', 'min', 'max'],
    'derivative_rmse': ['mean', 'std', 'min', 'max'],
    'trajectory_rmse': ['mean', 'std', 'min', 'max'],
    'model_complexity': ['mean', 'std', 'min', 'max'],
    'training_time': ['mean', 'std', 'min', 'max']
})
display(summary_stats)

In [ ]:
# Visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('SINDy Optimizer Ablation Study - Lorenz96 (1% Noise)', fontsize=16, fontweight='bold')

# Plot 1: Sparsity comparison
for optimizer in df_results['optimizer'].unique():
    data = df_results[df_results['optimizer'] == optimizer]
    axes[0, 0].plot(data['hyperparameter'], data['sparsity'], 'o-', label=optimizer, linewidth=2)
axes[0, 0].set_xlabel('Hyperparameter Value', fontsize=12)
axes[0, 0].set_ylabel('Sparsity', fontsize=12)
axes[0, 0].set_title('Sparsity vs Hyperparameter', fontsize=14)
axes[0, 0].set_xscale('log')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Derivative RMSE
for optimizer in df_results['optimizer'].unique():
    data = df_results[df_results['optimizer'] == optimizer]
    axes[0, 1].plot(data['hyperparameter'], data['derivative_rmse'], 'o-', label=optimizer, linewidth=2)
axes[0, 1].set_xlabel('Hyperparameter Value', fontsize=12)
axes[0, 1].set_ylabel('Derivative RMSE', fontsize=12)
axes[0, 1].set_title('Derivative Prediction Error', fontsize=14)
axes[0, 1].set_xscale('log')
axes[0, 1].set_yscale('log')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Trajectory RMSE
for optimizer in df_results['optimizer'].unique():
    data = df_results[df_results['optimizer'] == optimizer]
    axes[0, 2].plot(data['hyperparameter'], data['trajectory_rmse'], 'o-', label=optimizer, linewidth=2)
axes[0, 2].set_xlabel('Hyperparameter Value', fontsize=12)
axes[0, 2].set_ylabel('Trajectory RMSE', fontsize=12)
axes[0, 2].set_title('Trajectory Prediction Error', fontsize=14)
axes[0, 2].set_xscale('log')
axes[0, 2].set_yscale('log')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Plot 4: Model Complexity
for optimizer in df_results['optimizer'].unique():
    data = df_results[df_results['optimizer'] == optimizer]
    axes[1, 0].plot(data['hyperparameter'], data['model_complexity'], 'o-', label=optimizer, linewidth=2)
axes[1, 0].set_xlabel('Hyperparameter Value', fontsize=12)
axes[1, 0].set_ylabel('Model Complexity', fontsize=12)
axes[1, 0].set_title('Model Complexity vs Hyperparameter', fontsize=14)
axes[1, 0].set_xscale('log')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 5: Training Time
for optimizer in df_results['optimizer'].unique():
    data = df_results[df_results['optimizer'] == optimizer]
    axes[1, 1].plot(data['hyperparameter'], data['training_time'], 'o-', label=optimizer, linewidth=2)
axes[1, 1].set_xlabel('Hyperparameter Value', fontsize=12)
axes[1, 1].set_ylabel('Training Time (s)', fontsize=12)
axes[1, 1].set_title('Training Time vs Hyperparameter', fontsize=14)
axes[1, 1].set_xscale('log')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Plot 6: Pareto front (Sparsity vs RMSE)
for optimizer in df_results['optimizer'].unique():
    data = df_results[df_results['optimizer'] == optimizer]
    axes[1, 2].scatter(data['sparsity'], data['derivative_rmse'], label=optimizer, s=100, alpha=0.7)
axes[1, 2].set_xlabel('Sparsity', fontsize=12)
axes[1, 2].set_ylabel('Derivative RMSE', fontsize=12)
axes[1, 2].set_title('Pareto Front: Sparsity vs Error', fontsize=14)
axes[1, 2].set_yscale('log')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lorenz96_sindy_ablation_1percent_noise.png', dpi=300, bbox_inches='tight')
plt.show()

print("Visualization saved as 'lorenz96_sindy_ablation_1percent_noise.png'")

## Export Results to Excel

In [ ]:
# Export to Excel with multiple sheets
output_filename = 'lorenz96_sindy_ablation_1percent_noise.xlsx'

with pd.ExcelWriter(output_filename, engine='xlsxwriter') as writer:
    # Sheet 1: All results
    df_results.to_excel(writer, sheet_name='All Results', index=False)
    
    # Sheet 2: Summary statistics
    summary_stats.to_excel(writer, sheet_name='Summary Statistics')
    
    # Sheet 3: Best models per optimizer
    best_models = df_results.loc[df_results.groupby('optimizer')['derivative_rmse'].idxmin()]
    best_models.to_excel(writer, sheet_name='Best Models', index=False)
    
    # Sheet 4: Optimizer-specific results
    for optimizer in df_results['optimizer'].unique():
        optimizer_data = df_results[df_results['optimizer'] == optimizer]
        optimizer_data.to_excel(writer, sheet_name=f'{optimizer} Results', index=False)
    
    # Format the Excel file
    workbook = writer.book
    header_format = workbook.add_format({
        'bold': True,
        'text_wrap': True,
        'valign': 'top',
        'fg_color': '#D7E4BD',
        'border': 1
    })
    
    # Apply formatting to all sheets
    for sheet_name in writer.sheets:
        worksheet = writer.sheets[sheet_name]
        worksheet.set_column('A:Z', 18)

print(f"\nResults exported to '{output_filename}'")
print("\nExcel file contains the following sheets:")
print("  - All Results: Complete dataset with 1% noise")
print("  - Summary Statistics: Aggregated statistics by optimizer")
print("  - Best Models: Best performing model for each optimizer")
print("  - Individual optimizer sheets: Detailed results for each optimizer")

## Best Model Analysis

In [ ]:
# Find and display the best overall model
print("\nBest Models by Optimizer:")
print("="*80)

for optimizer in df_results['optimizer'].unique():
    optimizer_data = df_results[df_results['optimizer'] == optimizer]
    best_idx = optimizer_data['derivative_rmse'].idxmin()
    best_model = optimizer_data.loc[best_idx]
    
    print(f"\n{optimizer}:")
    print(f"  Hyperparameter: {best_model['hyperparameter']}")
    print(f"  Sparsity: {best_model['sparsity']:.4f} ({best_model['sparsity']*100:.2f}%)")
    print(f"  Derivative RMSE: {best_model['derivative_rmse']:.8f}")
    print(f"  Trajectory RMSE: {best_model['trajectory_rmse']:.8f}")
    print(f"  Model Complexity: {best_model['model_complexity']:.0f}")
    print(f"  Training Time: {best_model['training_time']:.4f}s")

# Overall best model
overall_best_idx = df_results['derivative_rmse'].idxmin()
overall_best = df_results.loc[overall_best_idx]

print("\n" + "="*80)
print("OVERALL BEST MODEL:")
print("="*80)
print(f"Optimizer: {overall_best['optimizer']}")
print(f"Hyperparameter: {overall_best['hyperparameter']}")
print(f"Sparsity: {overall_best['sparsity']:.4f} ({overall_best['sparsity']*100:.2f}%)")
print(f"Derivative RMSE: {overall_best['derivative_rmse']:.8f}")
print(f"Trajectory RMSE: {overall_best['trajectory_rmse']:.8f}")
print(f"Model Complexity: {overall_best['model_complexity']:.0f}")
print(f"Training Time: {overall_best['training_time']:.4f}s")

## Conclusion

This ablation study systematically compared four different SINDy optimizers (STLSQ, LASSO, Ridge, SR3) across various hyperparameter settings. The results provide insights into:

1. **Sparsity-Accuracy Trade-off**: How each optimizer balances model simplicity and prediction accuracy
2. **Computational Efficiency**: Training time comparison across optimizers
3. **Generalization**: Trajectory prediction performance on the same dataset
4. **Model Complexity**: Number of active terms and overall model structure

The comprehensive results are saved in the Excel file for further analysis.